# **Diabetes Prediction Project**

In [ ]:
import pandas as pd 
import numpy as np 

## **1 Data Loading**

In [2]:
df = pd.read_csv("diabetes_prediction_dataset.csv")
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


## **2 Data Understanding**

### 2.1 Columns Description

In [3]:
df.columns

Index(['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history',
       'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes'],
      dtype='object')

|Columns|Description|
|---|---|
| gender | Biological Sex of the patient |
| age | Age of the patient |
| hypertension | Whether the patient has hypertension (1) or not (0) |
| heart_disease | Whether the patient has heart disease (1) or not (0) |
|smoking_history| Smoking history of the patient.|
|bmi| BMI (Body Mass Index) is a measure of body fat based on weight and height.|
|HbA1c_level| (Hemoglobin A1C) HbA1c level is a measure of average blood sugar levels over the past 2-3 months.|
|blood_glucose_level| Blood glucose level is the amount of glucose present in the blood.|
|diabetes| Whether the patient has diabetes (1) or not (0) |

### 2.2 Feature Types

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  object 
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  object 
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.9+ MB


In [5]:
# Categorical Features
cat = df.select_dtypes(include="object").columns
print("Categorical Features: ", cat)
print("-"*50)

# Numerical Features
num = df.select_dtypes(exclude="object").columns.drop("diabetes")
print("Numerical Features: ", num)
print("-"*50)

# Target Variable
target = "diabetes"
print("Target Variable: ", target)

Categorical Features:  Index(['gender', 'smoking_history'], dtype='object')
--------------------------------------------------
Numerical Features:  Index(['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level',
       'blood_glucose_level'],
      dtype='object')
--------------------------------------------------
Target Variable:  diabetes


In [6]:
df["diabetes"].value_counts()/len(df)

diabetes
0    0.915
1    0.085
Name: count, dtype: float64

### 2.3 General Description

In [7]:
df.describe()

,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,diabetes
count,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,41.885856,0.07485,0.039420,27.320767,5.527507,138.058060,0.085000
std,22.516840,0.26315,0.194593,6.636783,1.070672,40.708136,0.278883
min,0.080000,0.00000,0.000000,10.010000,3.500000,80.000000,0.000000
25%,24.000000,0.00000,0.000000,23.630000,4.800000,100.000000,0.000000
50%,43.000000,0.00000,0.000000,27.320000,5.800000,140.000000,0.000000
75%,60.000000,0.00000,0.000000,29.580000,6.200000,159.000000,0.000000
max,80.000000,1.00000,1.000000,95.690000,9.000000,300.000000,1.000000


The ranges of the numbers are clear and logical, except for the extreme values ​​of "bmi" and "blood_glucose_level". However, these extreme values ​​do not have a strong effect on the difference between the median and the mean.

In [8]:
df.describe(include='object')

,gender,smoking_history
count,100000,100000
unique,3,6
top,Female,No Info
freq,58552,35816


### 2.4 Unique Values of Categorical Features

In [9]:
for c in cat:
    print(c)
    print(df[c].unique())
    print(df[c].nunique())
    print(df[c].value_counts())
    print("-"*50)

gender
['Female' 'Male' 'Other']
3
gender
Female    58552
Male      41430
Other        18
Name: count, dtype: int64
--------------------------------------------------
smoking_history
['never' 'No Info' 'current' 'former' 'ever' 'not current']
6
smoking_history
No Info        35816
never          35095
former          9352
current         9286
not current     6447
ever            4004
Name: count, dtype: int64
--------------------------------------------------


|Category|Meaning|
|---|---|
|never|Never smoked|
|No Info|No data available|
|ever|Smoked at some point (vague)|
|not current|Smoked in the past, not anymore|
|former|Was a regular smoker, now quit|
|current|Currently smoking|

In [10]:
df = df[df["gender"] != "Other" ].reset_index(drop=True)

## **3 Data Cleaning**

### 3.1 Missing Values

In [11]:
df.isnull().sum()

gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

### 3.2 Duplicated Values

In [12]:
df.duplicated().sum()

np.int64(3854)

In [13]:
df.drop_duplicates(inplace=True)

In [14]:
print("Duplicates:", df.duplicated().sum())
print("Missing Values:", df.isnull().sum().sum())

Duplicates: 0
Missing Values: 0


In [15]:
df.shape

(96128, 9)

## **4 Feature Engineering**

In [16]:
def categorize_age(age):
    if age < 12:
        return "Child"
    elif age < 18:
        return "Teen"
    elif age < 35:
        return "Young Adult"
    elif age < 60:
        return "Middle Aged"
    else:
        return "Senior"

def categorize_bmi(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Healthy"
    elif bmi < 30:
        return "Overweight"
    elif bmi < 40:
        return "Obese"
    else:
        return "Extremely Obese"

In [18]:
df["age_group"] = df["age"].apply(categorize_age)
df["bmi_cat"] = df["bmi"].apply(categorize_bmi)

In [19]:
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes,age_group,bmi_cat
0,Female,80.0,0,1,never,25.19,6.6,140,0,Senior,Overweight
1,Female,54.0,0,0,No Info,27.32,6.6,80,0,Middle Aged,Overweight
2,Male,28.0,0,0,never,27.32,5.7,158,0,Young Adult,Overweight
3,Female,36.0,0,0,current,23.45,5.0,155,0,Middle Aged,Healthy
4,Male,76.0,1,1,current,20.14,4.8,155,0,Senior,Healthy


In [20]:
df.to_csv("diabetes_prediction_dataset_cleaned.csv", index=False)